# RHIC p+p / O+O bulk tune: parameter exploration

Plan step A of the p+p / O+O bulk tune (the *Quick variant approach*, screening) and its follow-ups, on the
2 × RTX 3090 box. Events: `oo_bias.py` (groups `screen`, `var`, `grid`, `rem`; d+Au samples `dau_<set>`) through `oo_bias_docker.sh`; analysis:
`pp_bulk.py` (p+p against UA5), `oo_bulk.py --sample` (O+O against STAR), `centrality_norm.py` (how STAR's
centrality classes map onto the model's). This notebook only reads `results/`. Model results, not vetted physics
results.

```
./oo_bias_docker.sh --dev /data/OO-RHIC_bulk screen      # map, pp2407, m3     (2026-11-23, 2.3 h)
./oo_bias_docker.sh --dev /data/OO-RHIC_bulk var         # pp2407v03         (2026-11-23, 0.8 h)
./oo_bias_docker.sh --dev /data/OO-RHIC_bulk grid        # g_v*_n*  (4 sets)  (2026-11-23, 320 jobs, 3.1 h, no failures)
./oo_bias_docker.sh --dev /data/OO-RHIC_bulk rem         # α_rem scan on g_v045_n215, p+p (2026-11-23/24, 160 jobs, 1.4 h, no failures)
./oo_bias_docker.sh --dev /data/OO-RHIC_bulk dau_map dau_g_v045_n215   # d+Au (2026-11-24)
python refs/rhic_pp200/extract_rhic_pp200.py --phobos <arXiv:1011.1940 source>  # PHOBOS, STAR, PHENIX p+p
python pp_bulk.py analyse --out /data/OO-RHIC_bulk                      # -> results/pp_bulk.json
python oo_bulk.py analyse --sample oo_<set> --out /data/OO-RHIC_bulk     # -> results/oo_bulk_oo_<set>.json
python centrality_norm.py                                                # -> results/centrality_norm.json
python dau_bulk.py fetch && python dau_bulk.py analyse --out /data/OO-RHIC_bulk   # -> results/dau_bulk.json
```

## Why this exploration
The O+O runs used the bulk set of `config/OO_MCGlauber_MUSIC_isr_rhictune.xml` (the RHIC MAP of arXiv:2601.17234)
unchanged. Against STAR's O+O data (`oo_bulk_comparison.ipynb`) its multiplicity is too low, and more so the
more peripheral the class. Two explanations were open:

1. **The single-collision yield is too low.** The MAP's string rapidity loss at RHIC's beam rapidity
   ($y_\mathrm{init}\approx 5.36$) is 1.685; every set tuned to p+p or p+Pb has 2.15–2.38. Less rapidity loss means
   less string energy and fewer particles per nucleon–nucleon collision. **p+p tests this directly**: a p+p event
   is one collision.
2. **The multiple-collision side is wrong** (shadowing, the rapidity loss at low $y_\mathrm{init}$ of a nucleon's
   second and later collisions). p+p cannot test this; **O+O's multiplicity distribution can**.

Decision rules of the plan: p+p right and O+O right ⇒ adopt the set; p+p right but O+O off ⇒ multiple-collision
side, go to the full tune (B); p+p off for every set ⇒ also the hydro side (τ_form, e_sw, α_shift) first.

## The parameters (3dMCGlauber, `IS/MCGlauber` in the XML; method 4 unless noted)
| parameter | what it does |
|---|---|
| `ylossParam4At2/4/6/10` | mean rapidity loss of a string at $y_\mathrm{init}$ = 2, 4, 6, 10 (interpolated in between). At 200 GeV $y_\mathrm{beam}$ = 5.36: the first collision uses the At4–At6 interval, later collisions (lower $y_\mathrm{init}$) At2–At4 |
| `ylossParam4var` | variance of the event-by-event fluctuation of the rapidity loss (logit-normal): sets the width of P($N_\mathrm{ch}$) and, as found below, also the mean yield |
| `shadowing_factor` | reduces the rapidity loss of a nucleon's later collisions (acts only with more than one collision: not in p+p) |
| `BG` | width of the nucleon's gluon distribution ($B_G$, GeV$^{-2}$): the NN collision probability vs distance |
| `remnant_energy_loss_fraction` | $\alpha_\mathrm{rem}$: every wounded nucleon's remnant becomes a remnant string with rapidity loss = sampled loss × $\alpha_\mathrm{rem}$, deposited in MUSIC like the other strings; sets how far the remnants stop from beam rapidity, i.e. the forward $\eta$ region |
| method 3 (`rapidity_loss_method` 3) | rapidity loss as a power law in $y_\mathrm{init}$ (2211.16376: A 1.32, α1 1.8, α2 0.34, fluctuation variance 0.4) instead of nodes |

## The sets, and why each
| set | what | why |
|---|---|---|
| `map` | the O+O XML as is (2601.17234 MAP: At4–At6 ≈ 1.685, shadowing 0) | the baseline from the same machine |
| `pp2407` | 2407.17443's p+p set: nodes 1.6 / 2.15 / 2.15 / 2.15, var 0.5, shadowing 0.45, BG 5 | a set tuned to p+p (5.02 TeV) with the JETSCAPE chain |
| `m3` | method 3 (2211.16376), shadowing 0.45, α_rem 0.5 | the RHIC 3D-Glauber calibration (Au+Au, d+Au) |
| `pp2407v03` | `pp2407` with var 0.3 | screening: `pp2407`'s p+p P(n) ~10 % too wide and its O+O p($N_\mathrm{ch}$) tail too long — both say *too much fluctuation* |
| `g_v04_n215`, `g_v04_n225`, `g_v045_n215`, `g_v045_n225` | `pp2407` with var 0.4 / 0.45 × At4 = At6 = At10 = 2.15 / 2.25 | var 0.3 fixed the widths but lowered the yield by 8–9 % and cut the O+O tail too much: the optimum is between, with slightly more rapidity loss to restore the yield |

Every sample: 10,000 events in 40 jobs of 250, the same seeds for every set (p+p 740000+, O+O 730000+), so set-to-set
differences are less noisy than the statistical errors suggest. Minimum-bias stand-in (`--no-subtraction
--hard-pair-string-veto 0`, ISR chain with one hard scattering in 9–11 GeV whose hadrons are ignored), MUSIC + iSS
with one sample per event, **only the bulk (iSS) hadrons counted**. p+p: the O+O XML with projectile and target p,
b < 3 fm. O+O: b 0–10 fm as 3dMCGlauber samples it.

In [ ]:
import sys as _sys
_sys.path.insert(0, "..")
import _paths  # noqa: E402,F401  (analysis/OO-RHIC and its test directories on sys.path; shared results/, refs/, data/ in ..)
import json, sys
import numpy as np
import matplotlib.pyplot as plt
import tune as T
import oo_bias as B
RES = T.RES
S = json.loads((RES.parent / "refs" / "star_oo" / "star_oo_bulk.json").read_text())
PP = json.loads((RES / "pp_bulk.json").read_text())
CN = json.loads((RES / "centrality_norm.json").read_text())
ORDER = ["map", "pp2407", "m3", "pp2407v03"] + [s for s in B.BULK_SETS if s.startswith("g_")]
OO = {s: json.loads((RES / f"oo_bulk_oo_{s}.json").read_text()) for s in ORDER if (RES / f"oo_bulk_oo_{s}.json").exists()}
SETS = [s for s in ORDER if s in OO and f"pp_{s}" in PP["samples"]]
missing = [s for s in ORDER if s not in SETS]
print("sets with p+p and O+O results:", SETS)
print("not (yet) analysed:", missing or "none")
COL = {s: f"C{i}" for i, s in enumerate(ORDER)}

# what each set changes relative to the O+O XML (the MAP)
def overrides(s):
    return ", ".join(o.split("=", 1)[1].replace("IS/MCGlauber/", "") for o in B.BULK_SETS[s]) or "(none: the O+O XML)"
for s in ORDER:
    print(f"{s:12s} {overrides(s)}")

## 1. p+p 200 GeV against UA5: the single-collision yield
**Why:** p+p isolates one nucleon–nucleon collision, so it fixes the rapidity loss near $y_\mathrm{beam}$ and the
fluctuation width without the multiple-collision parameters. **Data:** UA5 (p–p̄ at 200 GeV; at mid-rapidity p+p
and p–p̄ agree within a few %), HEPData ins233599 (dN/dη NSD and inelastic) and ins267179 (NSD P(n) in |η| < 0.5,
1.5, 3, 5). PHOBOS p+p and STAR's p+p P($N_\mathrm{ch}$) are not on HEPData. **Model NSD:** ≥ 1 charged particle in
2 < |η| < 5.2 on both sides. Only bulk hadrons are counted; the beam remnants are part of the bulk (3dMCGlauber's
remnant strings are deposited in MUSIC), so |η| > 3 is a model prediction too. Caveats there: no diffraction in the
model, and hydro + Cooper–Frye at the low forward energy densities of p+p is the least certain part of the chain.

In [ ]:
ua = PP["data"]["dndeta_nsd"]
fig, axs = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
axs[0].errorbar(ua["eta"], ua["y"], ua["err"], fmt="ko", ms=4, label="UA5 NSD (p-pbar)")
for s in SETS:
    d = PP["samples"][f"pp_{s}"]["dndeta_nsd"]
    axs[0].plot(d["eta"], d["y"], "-", color=COL[s], label=s)
    axs[1].plot(ua["eta"], d["ratio_to_ua5"], "-", color=COL[s])
axs[0].set_ylabel("NSD dN$_{ch}$/dη"); axs[0].legend(fontsize=8, ncol=2); axs[0].set_ylim(0, 4)
axs[1].axhline(1, color="0.5", lw=0.8); axs[1].axvspan(3, 5, color="0.9", zorder=0)
axs[1].set_ylabel("model / UA5"); axs[1].set_xlabel("|η|"); axs[1].set_ylim(0.5, 1.4)
axs[1].text(3.1, 0.55, "forward: remnant strings in hydro, no diffraction", fontsize=8)
plt.tight_layout(); plt.show()

W = (0.5, 1.5, 3, 5)
print(f"{'set':12s} {'NSD |η|<0.5':>12s} {'inel':>6s} {'NSD frac':>8s}   <n>/UA5 at |η|< 0.5 1.5 3 5      width (σ/<n>) model/UA5 at |η|< 0.5 1.5 3 5")
for s in SETS:
    o = PP["samples"][f"pp_{s}"]
    mean = " ".join(f"{o[f'pn_{w:g}']['mean'] / o[f'pn_{w:g}']['ua5_mean']:.2f}" for w in W)
    wid = " ".join(f"{(o[f'pn_{w:g}']['disp'] / o[f'pn_{w:g}']['mean']) / (o[f'pn_{w:g}']['ua5_disp'] / o[f'pn_{w:g}']['ua5_mean']):.2f}" for w in W)
    print(f"{s:12s} {o['dndeta_nsd']['mid05']:7.2f}±{o['dndeta_nsd']['mid05_err']:.2f} {o['dndeta_inel']['mid05']:6.2f} {o['frac_nsd']:8.3f}   {mean:>24s}      {wid}")
o = PP["samples"][f"pp_{SETS[0]}"]
print(f"{'UA5':12s} {o['dndeta_nsd']['ua5_mid05']:7.2f}      {o['dndeta_inel']['ua5_mid05']:6.2f}")

### 1b. The other RHIC p+p data: PHOBOS over the full η range, STAR and PHENIX at mid-rapidity
**Why:** UA5 is p–p̄ and has its own normalisation; the RHIC experiments measured p+p at the same energy with
different triggers, and PHOBOS covers |η| ≤ 5.3, i.e. up to beam rapidity (5.36), where the beam remnants'
rapidity loss ($\alpha_\mathrm{rem}$) and the string ends decide the yield. None of these is on HEPData
(`refs/rhic_pp200/extract_rhic_pp200.py`):

- **PHOBOS** (arXiv:1011.1940, Fig. 28a): inelastic dN/dη read from the vector figure (54 points; their mean over
  |η| < 1 is 2.254, Table IV 2.25). The figure has no per-point errors, only a 90 % C.L. systematic band (≈ ±0.3,
  i.e. 8–15 % at 1σ, strongly correlated point to point); the two η halves are averaged, half their difference is
  shown as well. Compared up to |η| = 4.9 (the model's histogram ends at 5). PHOBOS's inelastic sample includes
  single diffraction, which the model does not generate. Left: absolute; right: the shape, both normalised to
  |η| < 1.
- **STAR** (arXiv:0808.2041, Table II): NSD (BBC coincidence 3.3 < |η| < 5.0, σ = 30.0 ± 3.5 mb), dN/dη at |η| < 0.5
  = 2.98 ± 0.34; the model with the same coincidence.
- **PHENIX** (arXiv:1509.06727, Sec. VII): minimum-bias p+p at mid-rapidity, 2.38 ± 0.17 (normalisation as stated
  there: minimum bias); shown against the model's inelastic and UA5-NSD values.

The three mid-rapidity values span 2.25 (PHOBOS, inelastic, |η| < 1) to 2.98 (STAR, NSD): about ±15 % around 2.5,
mostly normalisation. p+p fixes the yield level to that precision, not better.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
first = PP["samples"][f"pp_{SETS[0]}"]["rhic"]["phobos"]
pe, pd, ps = (np.array(first[k]) for k in ("eta", "data", "sys"))
pdiff = np.array(first["half_diff"])
mid_d = first["mid1_data"]
for col, norm in enumerate((False, True)):
    ax, rx = axs[0, col], axs[1, col]
    sc = 1 / mid_d if norm else 1.0
    ax.fill_between(pe, (pd - ps) * sc, (pd + ps) * sc, color="0.85", label="PHOBOS syst. (1σ)")
    ax.errorbar(pe, pd * sc, pdiff * sc, fmt="ko", ms=4, label="PHOBOS inel. (bars: ±η halves)")
    rx.fill_between(pe, 1 - ps / pd, 1 + ps / pd, color="0.85")
    for s in SETS:
        r = PP["samples"][f"pp_{s}"]["rhic"]["phobos"]
        m = np.array(r["model"]) * (1 / r["mid1_model"] if norm else 1.0)
        ax.plot(pe, m, "-", color=COL[s], label=s)
        rx.plot(pe, r["shape_ratio"] if norm else r["ratio"], "-", color=COL[s])
    ax.set_ylabel("dN$_{ch}$/dη" + (" / (|η|<1)" if norm else "") + "  (inelastic)")
    ax.set_title("absolute" if not norm else "shape, normalised at |η| < 1", fontsize=10)
    rx.axhline(1, color="0.5", lw=0.8); rx.set_ylim(0.5, 1.8); rx.set_xlabel("|η|"); rx.set_ylabel("model / PHOBOS")
axs[0, 0].legend(fontsize=8, ncol=2); axs[0, 0].set_ylim(0, 3.5)
plt.tight_layout(); plt.show()

print(f"{'set':12s} {'PHOBOS |η|<1':>13s} {'shape 3.1/4.1/4.9':>20s}   {'STAR NSD':>14s} {'(frac)':>7s}   {'PHENIX: inel/NSD':>17s}")
for s in SETS:
    r = PP["samples"][f"pp_{s}"]["rhic"]
    sh = " / ".join(f"{np.interp(x, r['phobos']['eta'], r['phobos']['shape_ratio']):.2f}" for x in (3.1, 4.1, 4.9))
    print(f"{s:12s} {r['phobos']['mid1_model']:6.2f}/{r['phobos']['mid1_data']:.2f}  {sh:>20s}   "
          f"{r['star']['model']:5.2f}±{r['star']['model_err']:.2f}/{r['star']['data']:.2f} {r['star']['frac_nsd_star']:7.3f}   "
          f"{r['phenix']['model_inel']:.2f}/{r['phenix']['model_nsd']:.2f} vs {r['phenix']['data']:.2f}")
print(f"data: STAR {r['star']['data']} ± {r['star']['err']} (NSD), PHENIX {r['phenix']['data']} ± {r['phenix']['err']} (min. bias)")

## 2. O+O: the shape of p($N_\mathrm{ch}$) — the test that does not depend on centrality
**Why:** STAR sorts events by $N_\mathrm{ch}$ itself ($0.2<p_T<2$ GeV, |η| < 1.5, efficiency corrected; arXiv:2510.19645
Fig. 5). Its measured p($N_\mathrm{ch}$) is depleted below $N_\mathrm{ch}\approx 20$ by the minimum-bias trigger, and
its centrality classes are fractions of a total that depends on how that is treated (section 4). The shape of
p($N_\mathrm{ch}$) above the trigger-affected region does not: model and STAR are both normalised to unit area above
$N_\mathrm{ch}$ = 21. Flat at 1 means the set makes the right number of particles *and* the right fluctuations in
O+O; a rising (falling) high-$N_\mathrm{ch}$ tail means too much (too little) fluctuation or too much (too little)
yield in central events.

In [ ]:
sx = np.array(S["flow_fig5a"]["nch"]); sp = np.array(S["flow_fig5a"]["p"])
NMIN = 21
def shape_ratio(s):
    p = OO[s]["pnch"]; e = np.array(p["edges"]); c = np.array(p["counts"], float); ce = np.array(p["counts_err"], float)
    mx = 0.5 * (e[1:] + e[:-1]); m = np.interp(sx, mx, c); me = np.interp(sx, mx, ce)
    use = sx >= NMIN; sc = sp[use].sum() / m[use].sum()
    return m * sc / sp, me * sc / sp, m[sx < 20].sum() * sc / sp[sx < 20].sum()
BANDS = [(21, 61), (61, 101), (101, 121), (121, 141), (141, 161), (161, 181)]
fig, ax = plt.subplots(figsize=(9, 4.5))
for s in SETS:
    r, re, _ = shape_ratio(s); m = (sx >= 11) & (sx < 181)
    ax.plot(sx[m], r[m], "-", color=COL[s], label=s)
    ax.fill_between(sx[m], (r - re)[m], (r + re)[m], color=COL[s], alpha=0.15)
ax.axhline(1, color="0.5", lw=0.8); ax.axvline(NMIN, color="0.6", ls=":"); ax.set_ylim(0, 2.5)
ax.set_xlabel("$N_{ch}$ (0.2 < $p_T$ < 2 GeV, |η| < 1.5)"); ax.set_ylabel("p($N_{ch}$) model / STAR\n(unit area above 21)")
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()
print(f"{'set':12s} <N_ch>  " + "  ".join(f"{a}-{b - 1:>3d}" for a, b in BANDS) + "   N_ch<20 model/STAR")
for s in SETS:
    r, re, low = shape_ratio(s)
    cells = []
    for a, b in BANDS:
        m = (sx >= a) & (sx < b)
        cells.append(f"{r[m].mean():.2f}±{np.sqrt((re[m] ** 2).sum()) / m.sum():.2f}")
    print(f"{s:12s} {OO[s]['pnch']['mean']:5.1f}  " + "  ".join(f"{c:>9s}" for c in cells) + f"   {low:.2f}")

## 3. O+O: $\langle N_\mathrm{ch}\rangle$ against centrality
STAR's Fig. 5b. Left: the model's classes as percentiles of all its events (every event with at least one NN
collision). Right: as fractions of the model's $f$ highest-multiplicity events, with $f$ chosen per set for the
flattest model/STAR ratio (`centrality_norm.py`). **Why both:** the left panel mixes the bulk physics with the
normalisation of the total cross section; if a single $f$ makes the ratio flat, the remaining offset is the yield and
the slope was the normalisation. (Fixed 2026-11-23: `oo_bulk.py` used to put the 80–100 % events into the 70–80 %
class, which made the last point drop to 0.24–0.45.)

In [ ]:
cl = np.array(CN["classes"]); cen = cl.mean(1); sn = np.array(S["flow_fig5b"]["nch"])
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for s in SETS:
    c = CN["samples"].get(f"oo_{s}")
    if c is None: continue
    axs[0].plot(cen, c["ratio_f1"], "o-", ms=3, color=COL[s], label=s)
    axs[1].plot(cen, c["ratio_flat_f"], "o-", ms=3, color=COL[s], label=f"{s}  (f = {c['flat_f']:.3f})")
for ax, t in zip(axs, ("percentiles of all model events", "percentiles of the f highest events")):
    ax.axhline(1, color="0.5", lw=0.8); ax.set_xlabel("centrality [%]"); ax.set_title(t, fontsize=10); ax.legend(fontsize=8)
axs[0].set_ylabel(r"$\langle N_{ch}\rangle$ model / STAR"); axs[0].set_ylim(0.3, 1.4)
plt.tight_layout(); plt.show()

## 4. Centrality normalisation: what STAR's classes are fractions of
**Why:** before tuning the multiple-collision parameters on the centrality dependence (step B, ~20 GPU hours here), check that the
centrality dependence is a bulk-physics observable at all. Three checks (`centrality_norm.py`):

1. **STAR alone.** STAR derives centrality from a Glauber + NBD fit to p($N_\mathrm{ch}$). Its Fig. 5b class means
   follow from its *measured* p($N_\mathrm{ch}$) (Fig. 5a) if the total area is the measured one plus a few %:
   the fit adds little below the trigger-depleted region, so STAR's classes are, in effect, fractions of the
   triggered sample. This also gives STAR's class edges in $N_\mathrm{ch}$.
2. **The model at STAR's edges.** The model centrality (fraction of model events above the edge) at STAR's
   $N_\mathrm{ch}$ edges, and at STAR's EPD edges (raw nMIP / $k$, $k$ fitted to the shape of STAR's minimum-bias
   nMIP above 78; arXiv:2604.13935 says its EPD classes are corrected for the trigger efficiency).
3. **STAR's trigger in the model.** STAR's minimum-bias trigger needs hits in TOF (|η| < 0.9) and in the forward
   detectors (2604.13935). Emulated from the bulk hadrons: TOF |η| < 1 plus a coincidence in the VPD (4.2 < |η| < 5.2)
   or EPD (2.2 < |η| < 5.0) acceptance. The remnants are in the bulk (remnant strings in MUSIC); the forward
   multiplicity, and so the VPD variant's efficiency, carries the model's forward uncertainty (no diffraction,
   hydro at low density). STAR's paper does not say which forward detector its O+O trigger used.

In [ ]:
st = CN["star"]
print(f"STAR: measured p(N_ch) = {100 * st['measured_frac_of_total']:.1f} % of the total that reproduces Fig. 5b "
      f"(chi2 {st['chi2']:.1f} for {st['n_classes']} classes at 2 %)")
print("STAR's N_ch edges:", ", ".join(f"{c} %: {v:.1f}" for c, v in st["nch_edges"].items() if c in ("1", "10", "20", "40", "60", "80")))
r = np.array(st["class_means_reproduced"]) / np.array(st["class_means_star"])
print("reproduced / STAR <N_ch> per class: " + " ".join(f"{x:.3f}" for x in r))
print()
print(f"{'set':12s}  model centrality at STAR's N_ch edges 10/20/40/60/80 %    at STAR's EPD edges (k)        f(flat)  trigger eff: TOF+EPD  TOF+VPD")
for s in SETS:
    c = CN["samples"].get(f"oo_{s}")
    if c is None: continue
    fe = " / ".join(f"{c['flow_edges'][k]['c_model']:4.1f}" for k in ("10", "20", "40", "60", "80"))
    ee = " / ".join(f"{c['epd_edges'][k]['c_model']:4.1f}" for k in ("0-10%", "10-20%", "20-40%", "40-60%", "60-80%"))
    tr = c["trigger"]
    print(f"{s:12s}  {fe:40s}  {ee} ({c['epd_k']:.2f})  {c['flat_f']:.3f}   {tr['tof1_epd']['eff']:.3f}      {tr['tof1_vpd']['eff']:.3f}")

fig, axs = plt.subplots(1, len(["tof1_epd", "tof2_epd", "tof1_vpd"]), figsize=(15, 3.8), sharey=True)
for ax, k in zip(axs, ("tof1_epd", "tof2_epd", "tof1_vpd")):
    for s in SETS:
        c = CN["samples"].get(f"oo_{s}")
        if c is None or k not in c["trigger"]: continue
        ax.plot(cen, c["trigger"][k]["ratio"], "o-", ms=3, color=COL[s], label=f"{s} ({c['trigger'][k]['eff']:.2f})")
    ax.axhline(1, color="0.5", lw=0.8); ax.set_title(f"trigger {k}", fontsize=10); ax.set_xlabel("centrality [%]")
axs[0].set_ylabel(r"$\langle N_{ch}\rangle$ model / STAR"); axs[0].legend(fontsize=7); axs[0].set_ylim(0.4, 1.5)
plt.tight_layout(); plt.show()

## 5. Summary per set, and the grid
One row per set, the numbers the decision uses: p+p yield and width against UA5, the O+O yield at the flattest
normalisation, the O+O p($N_\mathrm{ch}$) tail, and the EPD scale $k$ at STAR's class edges (flat = the forward
multiplicity distribution has STAR's shape). Then the 2 × 2 grid as tables in (variance, node level).

In [ ]:
def metrics(s):
    o = PP["samples"][f"pp_{s}"]; c = CN["samples"].get(f"oo_{s}", {})
    r, re, low = shape_ratio(s); tail = (sx >= 121) & (sx < 161); mid = (sx >= 21) & (sx < 101)
    k = OO[s]["epd"]["k_from_edges"]
    return {"pp NSD/UA5": o["dndeta_nsd"]["mid05"] / o["dndeta_nsd"]["ua5_mid05"],
            "pp width/UA5 (1.5)": (o["pn_1.5"]["disp"] / o["pn_1.5"]["mean"]) / (o["pn_1.5"]["ua5_disp"] / o["pn_1.5"]["ua5_mean"]),
            "OO <N>/STAR (flat f)": c.get("ratio_flat_f_mean", np.nan), "f": c.get("flat_f", np.nan),
            "OO p(N) 21-100": r[mid].mean(), "OO p(N) 121-160": r[tail].mean(),
            "EPD k 0-10/60-80": f"{k['0-10%']:.2f}/{k['60-80%']:.2f}"}
rows = {s: metrics(s) for s in SETS}
keys = list(next(iter(rows.values())).keys())
print(f"{'set':12s} " + " ".join(f"{k:>20s}" for k in keys))
for s, m in rows.items():
    print(f"{s:12s} " + " ".join(f"{v:20.3f}" if not isinstance(v, str) else f"{v:>20s}" for v in m.values()))

grid = [s for s in SETS if s.startswith("g_")]
if grid:
    def _val(s, key):
        return next(float(o.rsplit("=", 1)[1]) for o in B.BULK_SETS[s] if f"/{key}=" in o)
    par = {s: (_val(s, "ylossParam4var"), _val(s, "ylossParam4At4")) for s in grid}
    vs, ns = sorted({p[0] for p in par.values()}), sorted({p[1] for p in par.values()})
    for key in ("pp NSD/UA5", "pp width/UA5 (1.5)", "OO <N>/STAR (flat f)", "OO p(N) 121-160"):
        print(f"\n{key}:   rows var {vs}, columns nodes {ns}")
        for v in vs:
            print(f"  var {v:<5g}" + "".join(f"{rows[s][key]:9.3f}" if (s := next((g for g in grid if par[g] == (v, n)), None)) else "        -" for n in ns))
else:
    print("\ngrid: no results yet")

## 6. The remnants: $\alpha_\mathrm{rem}$ on the grid's best set
**Why:** every set is above PHOBOS beyond |η| ≈ 3.5 and below it at 1.5–3 (section 1b), with the shape normalised
at |η| < 1. The beam remnants decide the far-forward yield: 3dMCGlauber turns each wounded nucleon's remnant into a
string from its rapidity $y_\mathrm{rem}$ down to $y_\mathrm{rem} - \alpha_\mathrm{rem}\,y_\mathrm{loss}$
(`Glauber.cpp`, string mode −4), deposited in MUSIC. A larger $\alpha_\mathrm{rem}$ stretches the remnant strings toward
mid-rapidity. The XML's value is 0.536. The scan: `g_v045_n215` with $\alpha_\mathrm{rem}$ = 0.4 / 0.65 / 0.8 / 1.0
(sets `rem040` … `rem100`, group `rem`, p+p first, the same seeds), then O+O for the best value(s), because STAR's EPD
(2.1 < |η| < 5.1) sits in the same region and sets the centrality classes of the jet analysis.

In [ ]:
def arem(s):
    return next((float(o.rsplit("=", 1)[1]) for o in B.BULK_SETS[s] if "remnant_energy_loss_fraction" in o), 0.536)
REM = sorted(["g_v045_n215"] + [s for s in getattr(B, "REM_SETS", ()) if f"pp_{s}" in PP["samples"]], key=arem)
RCOL = {s: plt.cm.viridis(i / max(1, len(REM) - 1)) for i, s in enumerate(REM)}
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
first = PP["samples"][f"pp_{REM[0]}"]["rhic"]["phobos"]
pe, pd, ps = (np.array(first[k]) for k in ("eta", "data", "sys"))
for ax in axs[:2]:
    ax.fill_between(pe, 1 - ps / pd, 1 + ps / pd, color="0.88", label="PHOBOS syst. (1σ)")
ua = PP["data"]["dndeta_nsd"]
axs[2].fill_between(ua["eta"], 1 - np.array(ua["err"]) / np.array(ua["y"]), 1 + np.array(ua["err"]) / np.array(ua["y"]), color="0.88", label="UA5 error")
for s in REM:
    r = PP["samples"][f"pp_{s}"]
    lab = f"α_rem {arem(s):g}" + (" (XML)" if s == "g_v045_n215" else "")
    axs[0].plot(pe, r["rhic"]["phobos"]["ratio"], color=RCOL[s], label=lab)
    axs[1].plot(pe, r["rhic"]["phobos"]["shape_ratio"], color=RCOL[s])
    axs[2].plot(ua["eta"], r["dndeta_nsd"]["ratio_to_ua5"], color=RCOL[s])
for ax, t in zip(axs, ("model / PHOBOS (inelastic)", "shape: model / PHOBOS, both normalised at |η| < 1", "model / UA5 (NSD)")):
    ax.axhline(1, color="0.5", lw=0.8); ax.set_ylim(0.6, 1.6); ax.set_xlabel("|η|"); ax.set_title(t, fontsize=10)
axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

def shape_chi2(r, lo=1.0, hi=4.95):
    sh, eta = np.array(r["rhic"]["phobos"]["shape_ratio"]), np.array(r["rhic"]["phobos"]["eta"])
    m = (eta >= lo) & (eta <= hi)
    return float(np.mean(((sh[m] - 1) / (ps[m] / pd[m])) ** 2))
print(f"{'α_rem':>6s} {'NSD |η|<0.5':>12s} {'PHOBOS |η|<1':>13s}   shape at |η| 1.9 / 2.5 / 3.1 / 4.1 / 4.9   {'<pull²> 1-4.9':>13s}   UA5 NSD at 3.1 / 4.1   width/UA5 (1.5)")
for s in REM:
    r = PP["samples"][f"pp_{s}"]; ph = r["rhic"]["phobos"]
    sh = " / ".join(f"{np.interp(x, ph['eta'], ph['shape_ratio']):.2f}" for x in (1.9, 2.5, 3.1, 4.1, 4.9))
    u = " / ".join(f"{np.interp(x, ua['eta'], r['dndeta_nsd']['ratio_to_ua5']):.2f}" for x in (3.1, 4.1))
    w = (r["pn_1.5"]["disp"] / r["pn_1.5"]["mean"]) / (r["pn_1.5"]["ua5_disp"] / r["pn_1.5"]["ua5_mean"])
    print(f"{arem(s):6g} {r['dndeta_nsd']['mid05']:7.2f}±{r['dndeta_nsd']['mid05_err']:.2f} {ph['mid1_model']:9.2f}/{ph['mid1_data']:.2f}   {sh:>36s}   {shape_chi2(r):13.2f}   {u:>18s}   {w:.3f}")
print("<pull²>: mean squared (shape ratio − 1) / PHOBOS's relative 1σ band over 1 ≤ |η| ≤ 4.9; the band is strongly")
print("correlated point to point, so this ranks the values, it is not a χ² with a probability")

ROO = [s for s in REM if (RES / f"oo_bulk_oo_{s}.json").exists() and f"oo_{s}" in CN["samples"]]
if ROO:
    print(f"\nO+O: {'α_rem':>6s} {'<N>/STAR (f)':>14s} {'p(N) 121-160':>13s} {'EPD k 0-10/60-80':>17s} {'STAR EPD edges at model %':>28s}")
    for s in ROO:
        c = CN["samples"][f"oo_{s}"]; o = json.loads((RES / f"oo_bulk_oo_{s}.json").read_text())
        k = o["epd"]["k_from_edges"]; ed = " / ".join(f"{c['epd_edges'][l]['c_model']:.1f}" for l in c["epd_edges"])
        OO.setdefault(s, o)
        r_, re_, _ = shape_ratio(s); tail = (sx >= 121) & (sx < 161)
        print(f"      {arem(s):6g} {c['ratio_flat_f_mean']:8.3f} ({c['flat_f']:.3f}) {r_[tail].mean():13.2f} {k['0-10%']:8.2f}/{k['60-80%']:.2f} {ed:>28s}")

## 7. d+Au: the MAP's own calibration data
**Why:** the MAP (2601.17234) was calibrated on Au+Au (PHOBOS) and d+Au (PHENIX dN/dη, 0–20 %). The adopted set
changes the rapidity loss at beam rapidity (1.685 → 2.15), so d+Au must not break. **Data (HEPData):** PHENIX
(arXiv:1807.11928, ins1684475 Fig. 2b): dN/dη for |η| < 2.6 in six classes from the Au-going BBC (−3.9 < η < −3.1),
percentiles of the inelastic cross section; BRAHMS (ins643085 Table 2): minimum-bias dN/dη for −2 < η < 4.2, the
deuteron side reaching the remnant region; STAR (nucl-ex/0408016, ins656934 Table 2): dN/dη for |η| < 1 in minimum bias
and 0–20 / 20–40 / 40–100 % from the charged multiplicity in the Au-going FTPC (−3.8 < η < −2.8), percentiles of the
total cross section; and STAR's FTPC points at 3.0 ≤ |η| ≤ 3.5 on both sides (triangles; their classes from the charged
multiplicity at |η| < 0.5, so the model's FTPC-region values use that estimator), read from a ROOT figure of the
d+Au FTPC analysis (`refs/dau_200/extract_star_ftpc_dau.py`; the figure's 40–100 % and minimum-bias points are 13 % and
9 % above the published |η| < 1 values, so each class is scaled to the published mid-rapidity normalisation). **Model:** samples `dau_<set>` (`oo_bias.py`; d on Au, b < 12 fm, the same
chain and minimum-bias stand-in), classes from the model's own charged multiplicity in the Au-going BBC (PHENIX) or FTPC (STAR)
window (`dau_bulk.py`). The Au-going side is η < 0 in model and data.

In [ ]:
DP = RES / "dau_bulk.json"
if DP.exists():
    DA = json.loads(DP.read_text())["samples"]
    names = [n for n in ("dau_g_v045_n215", "dau_map") if n in DA]   # the scan sets: section 8
    labs = list(next(iter(DA.values()))["classes"])
    fig, axs = plt.subplots(2, 4, figsize=(16, 7))
    for i, lab in enumerate(labs):
        ax = axs.flat[i]; d0 = DA[names[0]]["classes"][lab]
        ax.errorbar(d0["eta"], d0["data"], d0["sys"], fmt="ko", ms=3, label="PHENIX")
        for j, n in enumerate(names):
            d = DA[n]["classes"][lab]
            ax.plot(d["eta"], d["model"], "s", ms=3, color=f"C{j}", label=n)
        ax.set_title(f"d+Au {lab}", fontsize=10); ax.set_xlabel("η (Au-going < 0)"); ax.set_ylim(0, None)
    axs.flat[0].legend(fontsize=8); axs.flat[0].set_ylabel("dN$_{ch}$/dη")
    ax = axs.flat[6]; b0 = DA[names[0]]["brahms_minbias"]
    o_ = np.argsort(b0["eta"]); b0 = {k: list(np.array(v)[o_]) for k, v in b0.items()}   # the table is not ordered in η
    srt = lambda v: np.array(v)[o_]
    ax.errorbar(b0["eta"], b0["data"], b0["err"], fmt="ko", ms=3, label="BRAHMS min. bias")
    for j, n in enumerate(names):
        ax.plot(b0["eta"], srt(DA[n]["brahms_minbias"]["model"]), "s", ms=3, color=f"C{j}", label=n)
    ax.set_title("d+Au minimum bias, BRAHMS", fontsize=10); ax.set_xlabel("η"); ax.legend(fontsize=7)
    ax = axs.flat[7]
    for j, n in enumerate(names):
        ax.plot(b0["eta"], srt(DA[n]["brahms_minbias"]["ratio"]), "s", ms=3, color=f"C{j}", label=n)
    ax.fill_between(b0["eta"], 1 - np.array(b0["err"]) / np.array(b0["data"]), 1 + np.array(b0["err"]) / np.array(b0["data"]), color="0.88")
    ax.axhline(1, color="0.5", lw=0.8); ax.set_ylim(0.5, 1.6); ax.set_title("model / BRAHMS", fontsize=10); ax.set_xlabel("η")
    plt.tight_layout(); plt.show()
    fig, axs = plt.subplots(1, 4, figsize=(16, 3.6), sharey=False)
    for i, lab in enumerate(DA[names[0]]["star"]):
        ax = axs[i]; d0 = DA[names[0]]["star"][lab]
        ax.errorbar(d0["eta"], d0["data"], d0["err"], fmt="ko", ms=3, label="STAR")
        for side in ("ftpc_au", "ftpc_d"):
            if side in d0:
                ax.errorbar(d0[side]["eta"], d0[side]["data"], d0[side]["err"], fmt="k^", ms=3)
        for j, n in enumerate(names):
            dn = DA[n]["star"][lab]
            ax.plot(d0["eta"], dn["model"], "s", ms=3, color=f"C{j}", label=n)
            for side in ("ftpc_au", "ftpc_d"):
                if side in dn:
                    ax.plot(dn[side]["eta"], dn[side]["model"], "s", ms=3, color=f"C{j}")
        ax.set_title(f"d+Au {lab} (STAR, FTPC-Au classes)", fontsize=10); ax.set_xlabel("η (Au-going < 0)"); ax.set_ylim(0, None)
    axs[0].legend(fontsize=8); axs[0].set_ylabel("dN$_{ch}$/dη"); plt.tight_layout(); plt.show()
    print(f"{'sample':22s} {'class':7s} {'events':>6s}   model/PHENIX: Au side (η<−1)   |η|<0.5   d side (η>1)   (PHENIX syst. at mid)")
    for n in names:
        for lab, d in DA[n]["classes"].items():
            print(f"{n:22s} {lab:7s} {d['n']:6d}   {d['ratio_au']:22.2f}   {d['ratio_mid']:7.2f}   {d['ratio_d']:12.2f}   ({100 * d['data_sys_rel_mid']:.0f} %)")
        o = DA[n]
        print(f"{n:22s} 0-20 %:  Au side {o['ratio_0_20_au']:.2f}, |η|<0.5 {o['ratio_0_20_mid']:.2f}, d side {o['ratio_0_20_d']:.2f}")
        for lab, d in o["star"].items():
            if "ftpc_au" in d:
                print(f"{n:22s} STAR {lab:8s} FTPC   model/STAR −3.5<η<−3 {d['ftpc_au']['ratio']:.2f}, 3<η<3.5 {d['ftpc_d']['ratio']:.2f}"
                      f"  (data {100 * d['ftpc_au']['data_err_rel']:.0f} / {100 * d['ftpc_d']['data_err_rel']:.0f} %)")
            print(f"{n:22s} STAR {lab:8s} {d['n']:6d}   model/STAR −1<η<−0.5 {d['ratio_au']:.2f}, |η|<0.5 {d['ratio_mid']:.2f}, 0.5<η<1 {d['ratio_d']:.2f}"
                  f"  (data {100 * d['err_rel']:.0f} %)   Au/d asymmetry model {d['asym_model']:.3f} vs {d['asym_data']:.3f}")
        print()
else:
    print("no d+Au results yet (dau_bulk.py analyse)")

## 8. Shadowing and the rapidity loss at small $y_\mathrm{init}$: the d+Au Au side and O+O
**Why:** the one clear miss of the adopted set is the Au-going side of d+Au (section 7: 1.16–1.33 × PHENIX at η < −1,
1.2–1.6 × STAR's FTPC-Au). Two parameters act on a nucleon's later collisions and were not scanned on the grid:
* `shadowing_factor` $s$ (0.45 in the adopted set, 0 in the MAP). In 3dMCGlauber (`Glauber::decide_produce_string_num`)
  a collision forms a string for sure if either nucleon has no string yet; only if *both* already have one does it form
  one with probability $(1-s)\,e^{-s(n-2)}$, $n$ the two nucleons' connections: 0.55 / 0.40 / 0.20 for $s$ = 0.45 / 0.6 / 0.8
  at $n$ = 2.
* `ylossParam4At2`, the mean rapidity loss at $y_\mathrm{init}$ = 2 (1.6 in the adopted set and the MAP): the node a
  nucleon reaches in its later collisions, after it has already lost rapidity (the deuteron's nucleons in d+Au). A
  smaller value leaves the deuteron's nucleons faster and moves particles from the Au side toward mid-rapidity and the
  d side.

**Sets** (`oo_bias.py` `SHAD_SETS`): `shad060`, `shad080` ($s$ = 0.6, 0.8) and `at2_110` (At2 = 1.1), each the adopted
set with one change; d+Au and O+O, 10,000 events each with the same seeds as `g_v045_n215` (same chain and
minimum-bias stand-in as sections 2–7). p+p is not affected (one collision per nucleon).

In [ ]:
SC = ["g_v045_n215", "shad060", "shad080", "at2_110"]
CS = {s: f"C{i}" for i, s in enumerate(SC)}
for s in SC:
    p = RES / f"oo_bulk_oo_{s}.json"
    if s not in OO and p.exists():
        OO[s] = json.loads(p.read_text())
DA = json.loads((RES / "dau_bulk.json").read_text())["samples"] if (RES / "dau_bulk.json").exists() else {}
SCD = [s for s in SC if f"dau_{s}" in DA]
SCO = [s for s in SC if s in OO and f"oo_{s}" in CN["samples"]]
print("d+Au:", SCD, "  O+O:", SCO)

# d+Au: model / data against eta, central and peripheral PHENIX classes and STAR minimum bias with the FTPC points
fig, axs = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
for ax, lab in zip(axs[:3], ("0-5%", "20-40%", "60-88%")):
    for s in SCD:
        d = DA[f"dau_{s}"]["classes"][lab]
        ax.plot(d["eta"], d["ratio"], "s-", ms=3, lw=0.8, color=CS[s], label=s)
    d = DA[f"dau_{SCD[0]}"]["classes"][lab]
    rel = np.array(d["sys"]) / np.array(d["data"])
    ax.fill_between(d["eta"], 1 - rel, 1 + rel, color="0.9", zorder=0)
    ax.set_title(f"d+Au {lab}, model / PHENIX", fontsize=10); ax.set_xlabel("η (Au-going < 0)")
ax = axs[3]
for s in SCD:
    d = DA[f"dau_{s}"]["star"]["minbias"]
    ax.plot(d["eta"], d["ratio"], "s-", ms=3, lw=0.8, color=CS[s], label=s)
    for side in ("ftpc_au", "ftpc_d"):
        if side in d:
            ax.plot(d[side]["eta"], np.array(d[side]["model"]) / np.array(d[side]["data"]), "^", ms=5, color=CS[s])
ax.set_title("d+Au minimum bias, model / STAR (▲ FTPC)", fontsize=10); ax.set_xlabel("η (Au-going < 0)")
for ax in axs:
    ax.axhline(1, color="0.5", lw=0.8)
axs[0].set_ylim(0.7, 1.8); axs[0].set_ylabel("model / data"); axs[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("d+Au, model / data")
print(f"{'set':12s} {'PHENIX 0-20 %: Au':>18s} {'mid':>5s} {'d':>5s}   {'60-88 % Au':>10s}   {'STAR FTPC-Au 0-20 / mb':>22s}   {'Au/d asym. 20-40 / mb':>22s}")
for s in SCD:
    o = DA[f"dau_{s}"]; st = o["star"]
    print(f"{s:12s} {o['ratio_0_20_au']:18.2f} {o['ratio_0_20_mid']:5.2f} {o['ratio_0_20_d']:5.2f}   {o['classes']['60-88%']['ratio_au']:10.2f}"
          f"   {st['0-20%']['ftpc_au']['ratio']:13.2f} / {st['minbias']['ftpc_au']['ratio']:.2f}"
          f"   {st['20-40%']['asym_model']:13.3f} / {st['minbias']['asym_model']:.3f}")
st = DA[f"dau_{SCD[0]}"]["star"]
print(f"{'STAR data':12s} {'':18s} {'':5s} {'':5s}   {'':10s}   {'':22s}   {st['20-40%']['asym_data']:13.3f} / {st['minbias']['asym_data']:.3f}")

# O+O: <N_ch> against centrality at the flattest normalisation and the shape of p(N_ch)
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for s in SCO:
    c = CN["samples"][f"oo_{s}"]
    axs[0].plot(cen, c["ratio_flat_f"], "o-", ms=3, color=CS[s], label=f"{s}  (f = {c['flat_f']:.3f})")
    r, re, _ = shape_ratio(s); m = (sx >= 11) & (sx < 181)
    axs[1].plot(sx[m], r[m], "-", color=CS[s], label=s)
    axs[1].fill_between(sx[m], (r - re)[m], (r + re)[m], color=CS[s], alpha=0.15)
axs[0].set_xlabel("centrality [%]"); axs[0].set_ylabel(r"$\langle N_{ch}\rangle$ model / STAR"); axs[0].set_ylim(0.8, 1.2)
axs[1].set_xlabel("$N_{ch}$ (0.2 < $p_T$ < 2 GeV, |η| < 1.5)"); axs[1].set_ylabel("p($N_{ch}$) model / STAR\n(unit area above 21)")
axs[1].axvline(NMIN, color="0.6", ls=":"); axs[1].set_ylim(0, 2.5)
for ax in axs:
    ax.axhline(1, color="0.5", lw=0.8); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("O+O")
print(f"{'set':12s} {'<N_ch> 0-1 %':>12s} {'flat f':>7s} {'<N>/STAR':>9s} {'rms log':>8s}   {'p(N) 101-120':>13s} {'121-160':>12s}   {'EPD k 0-10/60-80':>16s}")
for s in SCO:
    c = CN["samples"][f"oo_{s}"]; r, re, _ = shape_ratio(s); k = OO[s]["epd"]["k_from_edges"]
    cells = []
    for a, b in ((101, 121), (121, 161)):
        m = (sx >= a) & (sx < b)
        cells.append(f"{r[m].mean():.2f}±{np.sqrt((re[m] ** 2).sum()) / m.sum():.2f}")
    print(f"{s:12s} {OO[s]['nch_vs_cent']['nch'][0]:12.1f} {c['flat_f']:7.3f} {c['ratio_flat_f_mean']:9.3f} {c['flat_f_spread']:8.3f}"
          f"   {cells[0]:>13s} {cells[1]:>12s}   {k['0-10%']:>8.2f}/{k['60-80%']:.2f}")

## 9. The formation time: 0.5 or 0.424 fm/c
**Why:** the RHIC O+O XMLs (from X-SCAPE's `BulkFastTest/OO_one_event.xml`) and the p+p vacuum XML had
`tau_form_mean` 0.5 fm/c, the Au+Au, Pb+Pb and LHC O+O / Ne+Ne XMLs 0.424, the MAP's τ_form (2601.17234). 3dMCGlauber's
value is the effective one (MUSIC's `Initial_time_tau_0` is overwritten with initial profile 131). Is the tune valid for
either? Everything above ran with 0.5; since 2026-11-24 the tune XMLs have 0.424, so `tau0424` is the final XML.

**Set:** `tau0424` (`oo_bias.py`) = the adopted set with `tau_form_mean` 0.424, p+p, O+O and d+Au, 10,000 events each
with the same seeds as `g_v045_n215`.

In [ ]:
TS = ["g_v045_n215", "tau0424"]
for s in TS:
    p = RES / f"oo_bulk_oo_{s}.json"
    if s not in OO and p.exists():
        OO[s] = json.loads(p.read_text())
TS = [s for s in TS if f"pp_{s}" in PP["samples"] and s in OO and f"oo_{s}" in CN["samples"]]
print(f"{'set':12s} {'pp NSD |η|<0.5':>15s} {'PHOBOS shape 3.1/4.1/4.9':>25s} {'width/UA5':>9s}   {'OO <N_ch>':>9s} {'0-1 %':>6s} {'flat f':>6s} {'<N>/STAR':>8s}"
      f" {'p(N) 121-160':>12s} {'EPD k 0-10/60-80':>16s}")
for s in TS:
    o = PP["samples"][f"pp_{s}"]; c = CN["samples"][f"oo_{s}"]; ph = o["rhic"]["phobos"]
    sh = " / ".join(f"{np.interp(x, ph['eta'], ph['shape_ratio']):.2f}" for x in (3.1, 4.1, 4.9))
    wid = (o["pn_1.5"]["disp"] / o["pn_1.5"]["mean"]) / (o["pn_1.5"]["ua5_disp"] / o["pn_1.5"]["ua5_mean"])
    r, re, _ = shape_ratio(s); t = (sx >= 121) & (sx < 161); k = OO[s]["epd"]["k_from_edges"]
    print(f"{s:12s} {o['dndeta_nsd']['mid05']:9.2f}±{o['dndeta_nsd']['mid05_err']:.2f} {sh:>25s} {wid:9.3f}   {OO[s]['pnch']['mean']:9.1f}"
          f" {OO[s]['nch_vs_cent']['nch'][0]:6.1f} {c['flat_f']:6.3f} {c['ratio_flat_f_mean']:8.3f}"
          f" {r[t].mean():7.2f}±{np.sqrt((re[t] ** 2).sum()) / t.sum():.2f} {k['0-10%']:>8.2f}/{k['60-80%']:.2f}")

DT = [s for s in ("g_v045_n215", "tau0424") if f"dau_{s}" in DA]
if DT:
    print("\nd+Au, model / data")
    print(f"{'set':12s} {'PHENIX 0-20 %: Au':>18s} {'mid':>5s} {'d':>5s}   {'60-88 %: Au / mid':>17s}   {'STAR mb |η|<0.5':>15s}"
          f"   {'FTPC-Au / FTPC-d mb':>19s}   {'Au/d asym. 20-40 / mb':>22s}   {'BRAHMS η 3-4.2':>14s}")
    for s in DT:
        o = DA[f"dau_{s}"]; st = o["star"]; p8 = o["classes"]["60-88%"]; b = o["brahms_minbias"]
        bf = np.array(b["ratio"])[np.array(b["eta"]) >= 3].mean()
        print(f"{s:12s} {o['ratio_0_20_au']:18.2f} {o['ratio_0_20_mid']:5.2f} {o['ratio_0_20_d']:5.2f}   {p8['ratio_au']:10.2f} / {p8['ratio_mid']:.2f}"
              f"   {st['minbias']['ratio_mid']:15.2f}   {st['minbias']['ftpc_au']['ratio']:11.2f} / {st['minbias']['ftpc_d']['ratio']:.2f}"
              f"   {st['20-40%']['asym_model']:13.3f} / {st['minbias']['asym_model']:.3f}   {bf:14.2f}")

## 10. Identified $p_T$ spectra and $\langle p_T\rangle$: the hydro side
**Why:** the tune fixed yields and dN/dη through 3dMCGlauber; the shape of the $p_T$ spectra is set by MUSIC
(viscosities, the freeze-out at T = 0.15 GeV, radial flow) and by where the strings deposit their energy, none of which
was tuned. The stored bulk files keep every charged hadron's momentum and species, so π±, K±, p, p̄ come for free
(`pt_bulk.py`, no new events).

**Data:** STAR TOF (nucl-ex/0309012, HEPData ins628232): π, K, p spectra in NSD p+p and d+Au (minimum bias and
0–20 / 20–40 / 40–100 % from the Au-going FTPC), −0.5 < y < 0, $p_T$ 0.3–3 GeV. STAR dE/dx (arXiv:0808.2041, ins793126
Fig. 18): d+Au in the same classes, |y| < 0.1, $p_T$ 0.2–1.1 GeV; its ⟨$p_T$⟩ and dN/dy tables (VII, VIII, extrapolated;
p+p, d+Au, Au+Au 200 GeV) are transcribed in `refs/pt_200/`. STAR p+p in classes of the observed track multiplicity
$\hat n_\mathrm{ch}$ (|η| < 0.5, $p_T$ > 0.2 GeV, uncorrected; nucl-ex/0606028, ins719969): ⟨$p_T$⟩ by direct integration.
STAR's pions are corrected for weak decays, its protons are inclusive (≈ 20 % from hyperons).

**Model:** p+p with STAR's NSD condition (charged particles in both BBCs, 3.3 < |η| < 5); d+Au classes from the
Au-going FTPC multiplicity (as `dau_bulk.py`); p+p $\hat n_\mathrm{ch}$ as the count with $p_T$ > 0.2 GeV, |η| < 0.5,
thinned with STAR's efficiency (1/1.35); O+O in STAR's flow-paper classes at the flattest normalisation (no data).
**Caveats:** only the bulk hadrons — no (mini)jets, so above ≈ 1.5 GeV the model must fall below the data, most in
p+p; no hadronic afterburner (no late radial push); iSS has no weak decays, so model protons lack the hyperon
feed-down that STAR's include.

In [ ]:
PT = json.loads((RES / "pt_bulk.json").read_text()) if (RES / "pt_bulk.json").exists() else None
TAB = json.loads((RES.parent / "refs" / "pt_200" / "star_0808_2041_tables.json").read_text())
if PT is None:
    print("no results yet (pt_bulk.py analyse)")
else:
    SM = PT["samples"]; SPC = ["pi+", "K+", "p", "pi-", "K-", "pbar"]
    PSETS = [s for s in ("g_v045_n215", "tau0424", "map") if f"pp_{s}" in SM]
    PCOL = {"g_v045_n215": "C0", "tau0424": "C3", "map": "C1"}
    # spectra: p+p NSD and d+Au 0-20 % against STAR TOF (top: spectra, bottom: model / data)
    for sysn, lab, title in (("pp", "pp", "p+p NSD"), ("dau", "0-20%", "d+Au 0-20 %"), ("dau", "40-100%", "d+Au 40-100 %")):
        fig, axs = plt.subplots(2, 6, figsize=(18, 5.6), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
        for j, sp in enumerate(SPC):
            for s in PSETS:
                c = SM.get(f"{sysn}_{s}", {}).get("classes", {}).get(lab)
                if not c or sp not in c["tof"]:
                    continue
                t = c["tof"][sp]
                if s == PSETS[0]:
                    axs[0, j].errorbar(t["pt"], t["data"], t["data_err"], fmt="ko", ms=3, label="STAR TOF")
                axs[0, j].plot(t["pt"], t["model"], "s-", ms=3, lw=0.8, color=PCOL[s], label=s)
                axs[1, j].errorbar(t["pt"], t["ratio"], np.array(t["model_err"]) / np.array(t["data"]), fmt="s-", ms=3, lw=0.8, color=PCOL[s])
            axs[0, j].set_yscale("log"); axs[0, j].set_title(f"{title}: {sp}", fontsize=9)
            axs[1, j].axhline(1, color="0.5", lw=0.8); axs[1, j].set_ylim(0, 2); axs[1, j].set_xlabel("$p_T$ [GeV]")
        axs[0, 0].set_ylabel(r"$\frac{1}{N}\frac{d^2N}{2\pi p_T dp_T dy}$"); axs[1, 0].set_ylabel("model / STAR"); axs[0, 0].legend(fontsize=7)
        plt.tight_layout(); plt.show()

    # <pT> against the charged multiplicity: STAR p+p, d+Au, Au+Au 200 (Tables VII/VIII) and the model
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
    ti = {sp: TAB["species"].index(sp) for sp in TAB["species"]}
    for ax, sp in zip(axs, ("pi-", "K-", "pbar")):
        for grp, mk, lab in (("pp", "k*", "STAR p+p"), ("dAu", "kD", "STAR d+Au"), ("AuAu", "ko", "STAR Au+Au 200")):
            ks = [k for k in TAB["meanpt"] if k.split()[0] == grp]
            x = [sum(v[0] for v in TAB["dndy"][k]) for k in ks]
            y = [TAB["meanpt"][k][ti[sp]][0] for k in ks]; e = [TAB["meanpt"][k][ti[sp]][1] for k in ks]
            ax.errorbar(x, y, e, fmt=mk, ms=4, mfc="none", label=lab)
        for s in PSETS:
            for sysn, mk in (("pp", "*"), ("dau", "D"), ("oo", "o")):
                o = SM.get(f"{sysn}_{s}")
                if not o:
                    continue
                cl = [c for k, c in o["classes"].items() if not (sysn != "pp" and k == "minbias")]
                ax.plot([c["dnch_dy_6"] for c in cl], [c["meanpt"][sp][0] for c in cl], mk, ms=5, color=PCOL[s],
                        label=f"{s} ({sysn})" if sp == "pi-" else None)
        ax.set_xscale("log"); ax.set_xlabel(r"d$N_{ch}$/dy (π±, K±, p, p̄)"); ax.set_title(f"⟨$p_T$⟩ of {sp}", fontsize=10)
    axs[0].set_ylabel(r"$\langle p_T\rangle$ [GeV]"); axs[0].legend(fontsize=6, ncol=2); plt.tight_layout(); plt.show()

    # p+p <pT> against the observed multiplicity
    import pt_bulk as PTB
    DD = PTB.data()["ppmult"]["meanpt"]
    nh, di = np.array(DD["nhat"]), np.array(DD["direct integration"]["y"]); de = np.array(DD["direct integration"]["err"])
    ok = np.isfinite(di)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(nh[ok], di[ok], de[ok], fmt="ks", mfc="none", label="STAR, direct integration")
    xs = [1, 2, 3, 4, 5, 6, 7, 8, 9.5, 11.5]
    for s in PSETS:
        v = SM[f"pp_{s}"]["nhat"]
        ax.errorbar(xs, [x["meanpt"][0] for x in v.values()], [x["meanpt"][1] for x in v.values()], fmt="o-", ms=3, lw=0.8, color=PCOL[s], label=s)
    ax.set_xlabel(r"$\hat n_{ch}$ (tracks, $p_T$ > 0.2 GeV, |η| < 0.5)"); ax.set_ylabel(r"charged $\langle p_T\rangle$ [GeV]"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

    # tables: <pT> and dN/dy, model / STAR, and the particle ratios
    print(f"{'sample':18s} {'class':8s} " + " ".join(f"{'<pT> ' + sp:>11s}" for sp in ("pi-", "K-", "pbar")) + "   "
          + " ".join(f"{'dN/dy ' + sp:>11s}" for sp in ("pi-", "K-", "pbar")) + f"   {'K-/pi-':>11s} {'pbar/pi-':>11s}")
    for name, o in SM.items():
        for lab, c in o["classes"].items():
            tk = "pp" if o["system"] == "pp" else f"dAu {lab}" if o["system"] == "dau" else None
            def cell(q, sp):
                m = c[q][sp][0]
                return f"{m:.3f}/{TAB[q][tk][ti[sp]][0]:.3f}" if tk in TAB[q] else f"{m:.3f}      "
            kpi = c["dndy"]["K-"][0] / c["dndy"]["pi-"][0]; ppi = c["dndy"]["pbar"][0] / c["dndy"]["pi-"][0]
            dk = f"/{TAB['dndy'][tk][2][0] / TAB['dndy'][tk][0][0]:.3f}" if tk in TAB["dndy"] else "      "
            dp = f"/{TAB['dndy'][tk][4][0] / TAB['dndy'][tk][0][0]:.3f}" if tk in TAB["dndy"] else "      "
            print(f"{name:18s} {lab:8s} " + " ".join(f"{cell('meanpt', sp):>11s}" for sp in ("pi-", "K-", "pbar")) + "   "
                  + " ".join(f"{cell('dndy', sp):>11s}" for sp in ("pi-", "K-", "pbar")) + f"   {kpi:.3f}{dk} {ppi:.3f}{dp}")
    print("model / STAR; STAR's <pT> and dN/dy are extrapolated to all pT (|y| < 0.1), the model's are counted (|y| < 0.5).")
    print("Au+Au 200 for reference, K-/pi- 70-80 % / 0-5 %:", f"{TAB['dndy']['AuAu 70-80%'][2][0] / TAB['dndy']['AuAu 70-80%'][0][0]:.3f} /",
          f"{TAB['dndy']['AuAu 0-5%'][2][0] / TAB['dndy']['AuAu 0-5%'][0][0]:.3f}")

## Results and what they mean
2026-11-23/24, 10,000 p+p and 10,000 O+O (or d+Au) events per set, statistical errors only, bulk hadrons only. Model results, not
vetted physics results.

1. **p+p confirms the single-collision hypothesis.** The MAP is 30 % low at mid-rapidity (NSD dN/dη 1.74 ± 0.02
   against UA5's 2.50; 0.73 × PHOBOS inelastic). `pp2407` matches UA5 (2.52 ± 0.02), `m3` is 7 % high (2.66). Every
   set with p+p-tuned rapidity loss gets the yield; the MAP's 1.685 near $y_\mathrm{beam}$ is the main cause of the
   O+O deficit. The RHIC data bracket the level: PHOBOS inelastic 2.25 (|η| < 1), PHENIX 2.38 ± 0.17, STAR NSD
   2.98 ± 0.34 — `pp2407` (inelastic 2.47 at |η| < 1, STAR-like NSD 2.49) is 10 % above PHOBOS and 1.4σ below STAR;
   p+p fixes the yield to about ±15 %, mostly the experiments' normalisations.
1b. **The forward region is too high, and the remnants are the handle.** Normalised at |η| < 1, PHOBOS's shape
   gives model/data 1.11 / 1.17 at |η| = 4.1 / 4.9 for `pp2407` and 1.05 / 1.08 for `m3` (α_rem 0.5), 1.42 / 1.87 for
   the MAP, and 1.22 / 1.30 with variance 0.3 (smaller fluctuations leave more energy at forward rapidity); UA5 says
   the same (1.2 at |η| ≈ 4.1). PHOBOS's band is ≈ 10–15 % (1σ) there, so this is a 1σ effect for `pp2407`/`m3` and
   a clear one for the MAP. The shape is shifted, not only high at the edge: every set is 10–15 % below PHOBOS at
   |η| = 1.5–3 (PHOBOS's shoulder at |η| ≈ 1.5 is missing) and above it beyond 3.5, with the same pattern for all
   sets — more particles near the string ends and remnants, fewer in between. The beam remnants are in the model (3dMCGlauber's remnant strings, rapidity loss ×
   α_rem, deposited in MUSIC); α_rem is the next parameter to scan once variance and nodes are fixed. Caveats: no
   diffraction in the model; forward hydro at low density.
2. **The O+O "slope" is mostly the centrality normalisation, not the multiple-collision physics.** With the model's
   classes as percentiles of all its events, every set falls toward peripheral (`pp2407`: 1.02 at 0–1 % to 0.66 at
   70–80 %). STAR's own Fig. 5b follows from its measured, trigger-depleted p($N_\mathrm{ch}$) plus only 2.9 % of area,
   so STAR's classes are fractions of the triggered sample. Taking the model's classes as fractions of its highest 85.5 %
   makes `pp2407` flat at 1.04 (rms 0.7 %) from 0–1 % to 70–80 %; `m3` and `pp2407v03` need the same $f$ (0.86–0.865), only
   the MAP (whose shape is wrong) a different one. The emulated STAR trigger brackets it: TOF + EPD coincidence keeps
   93–97 %, TOF + VPD 73–74 % (forward model uncertainty, see section 4). So the multiple-collision side need not be retuned on
   this observable; the plan's step B is not needed for it.
3. **What is left is the fluctuation width.** `pp2407`'s p+p P(n) is 5–13 % too wide and its O+O p($N_\mathrm{ch}$) tail
   above 120 is 1.2–2 × STAR's. Variance 0.3 (`pp2407v03`) fixes the p+p widths exactly (0.93 / 0.74 / 0.64 / 0.49
   against 0.95 / 0.75 / 0.63 / 0.54) but lowers the yield by 8–9 % in p+p and O+O and cuts the O+O tail to 0.2–0.8. The
   variance is not only a width: it shifts the mean yield too, so variance and node level are tuned together — the grid.
4. **The EPD agrees** with the normalisation picture: $k$ at STAR's class edges is flat for the p+p-tuned sets
   (2.47–2.63) but rises for the MAP (2.76 → 3.05), and STAR's EPD edges sit at model centralities 8 / 18 / 37 / 56 / 76 %
   for 10 / 20 / 40 / 60 / 80 % (`pp2407`).
5. **Consequences for the jet samples (R4b, I_CP).** Model classes must use STAR's normalisation: for $N_\mathrm{ch}$
   classes STAR's edges in $N_\mathrm{ch}$ (section 4), for the EPD classes the model's percentiles or STAR's nMIP
   edges / $k$; the difference between the two is the class-edge systematic (STAR varies its EA edges by ±5 %
   relative). The bias samples (`mb`, `jets`, ...) are redone with the adopted bulk set.

6. **The grid: the node level stays at 2.15, the variance goes to 0.45.**
   * The node level ($y_\mathrm{loss}$ at $y_\mathrm{init}$ = 4, 6, 10) is the strong knob. From 2.15 to 2.25 the p+p and
     O+O yields rise by 11–12 % and the O+O tail above 120 by a factor ≈ 1.8. Both 2.25 sets overshoot everything:
     p+p NSD 7–9 % above UA5, O+O ⟨$N_\mathrm{ch}$⟩ 8–10 % above STAR, tail 1.7–1.9 × STAR's.
   * At nodes 2.15, the variance moves everything together and monotonically (variance 0.3 / 0.4 / 0.45 / 0.5):

     | | 0.3 | 0.4 | 0.45 | 0.5 (`pp2407`) |
     |---|---|---|---|---|
     | p+p NSD / UA5 | 0.92 | 0.95 | 0.98 | 1.01 |
     | p+p P(n) width / UA5 (\|η\| < 1.5) | 0.98 | 1.05 | 1.07 | 1.10 |
     | O+O ⟨$N_\mathrm{ch}$⟩ / STAR (flat $f$) | 0.95 | 0.99 | 1.01 | 1.04 |
     | O+O p($N_\mathrm{ch}$) 121–160 / STAR | 0.57 | 0.88 | 1.04 | 1.32 |

     No variance gets the p+p width and the yield exactly right together: the width is right at 0.3, where the yield
     is 8 % low.
   * **Candidate to adopt: `g_v045_n215`** (`pp2407` with `ylossParam4var` 0.45). Against data:
     * O+O: ⟨$N_\mathrm{ch}$⟩ 1.01 × STAR, flat from 0–1 % to 70–80 % with the same normalisation as before
       ($f$ = 0.855); p($N_\mathrm{ch}$) at 101–120 / 121–140 / 141–160 is 1.12 ± 0.06 / 1.12 ± 0.09 / 0.96 ± 0.15
       of STAR's; EPD $k$ flat (2.48–2.56).
     * p+p: NSD 2.44 ± 0.02 (2.5 % below UA5's 2.50), PHENIX 2.36 / 2.44 (inelastic / NSD) against 2.38 ± 0.17,
       PHOBOS 2.40 against 2.25 (+7 %), STAR 2.38 against 2.98 ± 0.34 (−1.8σ). The p+p width is still 7 % above UA5's.
     * `g_v04_n215` is the neighbour: 2–3 % lower yield, 3 % narrower p+p width, O+O tail 0.88. It is close
       enough to serve as the systematic variation.
   * The normalisation does not depend on the set: every p+p-tuned set needs $f$ = 0.855–0.865 and gets the same trigger
     efficiencies.
   * The forward shape does not change with variance or nodes beyond what the yield does (`g_v045_n215`, normalised at
     |η| < 1: 0.91 / 1.13 / 1.19 × PHOBOS at |η| = 3.1 / 4.1 / 4.9), so α_rem remains the open handle there.

7. **The remnants do not fix the forward shape; α_rem stays at 0.536** (section 6; 2026-11-24, 10k p+p per value).
   * A larger α_rem does not spread the remnant particles: it moves a narrow bump inward, from |η| ≈ 4.5–5 at 0.4 to
     ≈ 3 at 1.0. No value removes both the deficit at |η| 1.5–3 and the excess beyond 3.5.
   * Agreement with PHOBOS's shape (mean squared pull over 1 ≤ |η| ≤ 4.9, in units of its 1σ band): 1.87 / **0.87** /
     1.21 / 4.35 / 18.8 for α_rem 0.4 / **0.536** / 0.65 / 0.8 / 1.0. The XML's value is at the minimum. The
     mid-rapidity yield hardly moves (NSD 2.54 / 2.44 / 2.42 / 2.40 / 2.34).
   * A broader remnant deposit would need the remnant fraction to fluctuate. The XML keys for that
     (`fluctuation_remnant_energy_loss_fraction`, `remnant_energy_loss_fraction_var`) are passed by X-SCAPE but read by
     no 3dMCGlauber version (upstream branches, the pinned commits of every X-SCAPE branch, the `oo_rhic_run` fork):
     it would be new code in `Glauber.cpp`.
8. **d+Au: the candidate set does not break the MAP's calibration data; at mid-rapidity it is closer to it than the
   MAP in this chain** (section 7; 10k minimum-bias events per set).
   * The MAP, in our chain, is 15–18 % low at mid-rapidity in d+Au: PHENIX 0–20 % 0.84 (data syst. 9–16 %), STAR 0–20 %
     0.81 and minimum bias 0.82 (data 4–5 %), BRAHMS minimum bias 0.85. The MAP was fitted to PHENIX's 0–20 %, so our
     chain does not reproduce the paper's own calibration point (freeze-out at T = 0.15 GeV instead of e_sw + UrQMD,
     α_shift = 0, the 3dMCGlauber version) — the same direction as its p+p and O+O deficit.
   * `g_v045_n215`: mid-rapidity 1.02–1.09 of PHENIX in 0–20 % (1.07 for 0–20 %), 1.03 of STAR in 0–20 %, 1.07 of
     STAR minimum bias, about 1.0–1.1 of BRAHMS; it rises toward peripheral (PHENIX 1.12 / 1.16 / 1.23 at 20–40 /
     40–60 / 60–88 %, STAR 20–40 % 1.11; STAR 40–100 % 1.05).
   * The deuteron side, where the deuteron's remnants land, is right: PHENIX η > 1 at 0.98–1.08, STAR FTPC-d
     (3 < η < 3.5) 1.01 / 0.99 / 1.19 / 1.10, BRAHMS up to η = 4.2 at 0.96–1.13.
   * **The Au-going side is too high**: PHENIX η < −1 at 1.16–1.33, STAR FTPC-Au (−3.5 < η < −3) 1.19 / 1.34 / 1.57 /
     1.39 (0–20 / 20–40 / 40–100 % / minimum bias), more toward peripheral. The MAP has the same pattern relative to
     its mid-rapidity (Au side about 10 % above mid relative to data), so it is a feature of the chain's Au-side
     production (the Au participants' strings, shadowing), amplified by the higher yield. STAR's Au/d asymmetry at
     |η| < 1: model 1.30 vs 1.23 (0–20 %), 1.20 vs 1.17 (minimum bias).
   * Caveats: the FTPC points come from a 2004 figure (40–100 % and minimum bias rescaled to the published
     normalisation); model classes as percentiles of all its events (no diffraction); no p_T cut on the model's class
     estimators.

9. **Shadowing does not fix the d+Au Au side; the shadowing factor stays at 0.45** (section 8; 2026-11-24, 10k d+Au
   and 10k O+O per set).
   * In d+Au, shadowing hardly acts. It only applies to a collision of two nucleons that both already have a string.
     A Au nucleon meets at most the two deuteron nucleons, so most collisions involve a fresh Au nucleon and always
     form a string. $s$ = 0.6 / 0.8 move the Au side by at most 2 % (PHENIX 0–20 %: 1.16 / 1.17 against 1.18; STAR
     FTPC-Au minimum bias 1.38 / 1.39 against 1.39), within statistics.
   * In O+O, where both nucleons are often already connected, a higher $s$ removes strings from the most central
     events: ⟨$N_\mathrm{ch}$⟩ at 0–1 % 146.0 / 140.6 against 148.5, the p($N_\mathrm{ch}$) tail at 121–160 0.99 ± 0.08 /
     0.81 ± 0.07 of STAR's against 1.04 ± 0.09, and $s$ = 0.8 needs a different normalisation ($f$ = 0.880) and is
     still 2–4 % low at 0–10 %. So O+O prefers the adopted 0.45.
   * `ylossParam4At2` 1.1 (from 1.6) is the only change that moves the d+Au Au side: PHENIX 0–20 % 1.14 against 1.18
     (≈ 2σ, same seeds), STAR's Au/d asymmetry at 20–40 % 1.160 against 1.210 (data 1.152), minimum bias 1.175 against
     1.201 (data 1.165). It is neutral in O+O (flat at 1.004, $f$ = 0.855, tail 1.08 ± 0.09). The excess stays at
     ≈ 1.14 in central and 1.33 in peripheral collisions (60–88 %), and the FTPC-Au points at 1.17–1.38.
   * Together with the class-fraction test (`dau_bulk.py --class-frac`) and the grid, none of the bulk parameters
     tried (variance, nodes, α_rem, shadowing, At2, the class normalisation) removes the Au-side excess. It stays a
     known miss of the chain. Candidates beyond these parameters: the Au participants' string ends (the target
     side's rapidity loss), freeze-out (T = 0.15 GeV, no UrQMD), and the model's class estimators (no p_T cut).

10. **The formation time does not matter at the 1–2 % level; the XMLs switch to the MAP's 0.424 fm/c** (section 9;
    2026-11-24, 10k p+p, O+O and d+Au each).
    * `tau_form_mean` 0.424 instead of 0.5: p+p NSD 2.48 ± 0.02 against 2.44 (+1.6 %), the PHOBOS forward shape and the
      p+p width unchanged; O+O ⟨$N_\mathrm{ch}$⟩ 37.3 against 37.5, yield against STAR at the flattest normalisation 1.003 against 1.014
      ($f$ 0.860 against 0.855), tail 1.01 ± 0.08 against 1.04 ± 0.09, EPD $k$ unchanged; d+Au within 1 % everywhere
      (PHENIX 0–20 % Au side / mid / d side 1.18 / 1.06 / 0.99 against 1.18 / 1.07 / 0.99, STAR FTPC-Au minimum bias 1.39
      in both, BRAHMS within 3 %); ⟨$p_T$⟩ ≤ 2 % (section 10).
    * A Bjorken estimate (fixed energy density at τ₀, $S \propto \tau_0^{1/4}$) predicted ≈ 4 % fewer particles. Here
      the strings deposit a fixed energy as sources in MUSIC, so the start time only shifts when it enters; the
      measured change is smaller and of opposite sign in p+p.
    * In the vacuum chain τ_form has no effect at all (same-seed test: identical hadrons). So the tune holds for
      either value. Since 2026-11-24 the tune XMLs have 0.424 (second revision note in their headers): the MAP's
      τ_form in the parameter that sets it, as in the Au+Au, Pb+Pb and LHC O+O / Ne+Ne XMLs. `tau0424` is the final
      XML; the samples above ran with 0.5, which `oo_bias.py` now pins for them.

11. **The $p_T$ shapes show what the tune could not: no minijets, no hadronic afterburner, and no strangeness
    suppression** (section 10; 2026-11-24, the existing samples, no new events).
    * ⟨$p_T$⟩ is low for every species, by about 5 % for pions (p+p 0.329 against STAR's 0.348, d+Au 0–20 % 0.361
      against 0.378), 20 % for kaons (0.41–0.48 against 0.52–0.62) and 15–20 % for antiprotons (0.58–0.70 against
      0.68–0.90), with the same small rise with multiplicity as the data but at too low a level. The bulk set hardly
      matters (MAP within 1–3 %), nor does the formation time (0.424 vs 0.5: ≤ 2 %). Missing late radial push (no
      afterburner) and the missing hard component both lower it; the hydro parameters are the MAP's.
    * Spectra: below ≈ 0.7 GeV within 0.7–1.3 of STAR; above 1 GeV the bulk falls far below (p+p pions 0.1–0.2 at
      1–2 GeV, d+Au 0.2–0.25), as expected without (mini)jets — the hard part of these events is the jet sector's.
    * Chemistry: kaons are 40–50 % too many (K⁻/π⁻ 0.17–0.18 in p+p, d+Au and O+O against STAR's 0.10 in p+p and
      0.12–0.13 in d+Au; Au+Au 0–5 % has 0.15): iSS samples a grand-canonical ensemble, so the canonical strangeness
      suppression of small systems is missing. p+p pions are 16 % low (1.17 against 1.42), so the charged yield the
      tune matched is partly kaons. p̄/π⁻ 0.095 against 0.080 (p+p) and 0.094 (d+Au 0–20 %) is about right, given that
      STAR's antiprotons include hyperon feed-down the model's do not.
    * The p+p charged ⟨$p_T$⟩ against STAR's observed multiplicity agrees within 1–5 % (0.373 → 0.455 against 0.393 →
      0.465 for $\hat n_\mathrm{ch}$ = 1 … 11.5), but for partly the wrong reasons: identified ⟨$p_T$⟩ low, kaon
      fraction high, and a hydro rise instead of STAR's minijets.
    * Consequences: none for the hand tune (yields and dN/dη) or for the jet observables, which take their hard part
      from the jet sector. For flow and identified-particle observables, the hydro and the particlization (canonical
      strangeness, an afterburner) are the open items; the MAP's hydro was calibrated with UrQMD.

**Hand tune complete.** The final XML is `g_v045_n215` with τ_form 0.424 (α_rem 0.536, shadowing 0.45,
`ylossParam4At2` 1.6), validated as `tau0424` in p+p, O+O and d+Au; no other change after the revision needed a run, and the hard tune holds with it
(only BG reaches the vacuum chain; the reference point gives χ² 35.4 against 33.8 over the 35 fit bins with the old
block, the UE the same). `ylossParam4At2` 1.1 stays an option for d+Au (≈ 3 % on the Au side). **Next:** freeze the
XMLs (commit and image tag), then the physics run's `mb` (50k, it doubles as the O+O validation), `jets_vac`, and
`jets_med`. The yields here count only bulk hadrons.